In [ ]:
from google.colab import userdata
import os

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('YC_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('YC_SECRET_ACCESS_KEY')
S3_BUCKET   = userdata.get('S3_BUCKET')
S3_ENDPOINT = 'https://storage.yandexcloud.net'

In [ ]:
os.system('pip install awscli transformers scikit-learn sentencepiece -q')
os.system(f'aws s3 sync s3://{S3_BUCKET}/second/nlp/dataset_processed/ /content/dataset_processed/ --endpoint-url {S3_ENDPOINT}')
print('dataset ready')

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score

In [ ]:
PROCESSED_DIR = Path('/content/dataset_processed')

RUN           = 'roberta_3'
MODEL_NAME    = 'roberta-base'
WEIGHTS_DIR   = Path(f'/content/weights/{RUN}')
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE    = 16
GRAD_ACCUM    = 2
EPOCHS        = 6
LR            = 1e-5
WEIGHT_DECAY  = 2e-2
MAX_LEN       = 256
WARMUP_RATIO  = 0.1
LABEL_SMOOTH  = 0.15
MAX_GRAD_NORM = 1.0
LLRD_FACTOR   = 0.9
PATIENCE      = 3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}  run: {RUN}  effective_batch: {BATCH_SIZE * GRAD_ACCUM}')

In [ ]:
from torch.utils.data import WeightedRandomSampler

with open(PROCESSED_DIR / 'label_map.json') as f:
    label_map = {int(k): v for k, v in json.load(f).items()}

NUM_LABELS = len(label_map)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, path: Path):
        df = pd.read_csv(path)
        self.texts  = df['text'].astype(str).tolist()
        self.labels = df['label'].tolist()

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_set = TextDataset(PROCESSED_DIR / 'train.csv')
test_set  = TextDataset(PROCESSED_DIR / 'test.csv')

counts = torch.zeros(NUM_LABELS)
for lbl in train_set.labels:
    counts[lbl] += 1
class_weights = (counts.sum() / (NUM_LABELS * counts)).to(DEVICE)

sample_weights = [class_weights[lbl].item() for lbl in train_set.labels]
sampler        = WeightedRandomSampler(sample_weights, num_samples=len(train_set), replacement=True)
train_loader   = DataLoader(train_set, batch_size=BATCH_SIZE,     sampler=sampler,  num_workers=2, pin_memory=True)
test_loader    = DataLoader(test_set,  batch_size=BATCH_SIZE * 2, shuffle=False,    num_workers=2, pin_memory=True)

print(f'train: {len(train_set)}  test: {len(test_set)}')
print('class weights:', {label_map[i]: f'{class_weights[i]:.3f}' for i in range(NUM_LABELS)})

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(DEVICE)

no_decay = ['bias', 'LayerNorm.weight']
layers   = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
layers.reverse()

param_groups = []
lr = LR
for layer in layers:
    param_groups += [
        {'params': [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)], 'lr': lr, 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in layer.named_parameters() if     any(nd in n for nd in no_decay)], 'lr': lr, 'weight_decay': 0.0},
    ]
    lr *= LLRD_FACTOR

param_groups += [
    {'params': [p for n, p in model.classifier.named_parameters() if not any(nd in n for nd in no_decay)], 'lr': LR, 'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.classifier.named_parameters() if     any(nd in n for nd in no_decay)], 'lr': LR, 'weight_decay': 0.0},
]

optimizer    = torch.optim.AdamW(param_groups)
total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler    = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
criterion    = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

print(f'params: {sum(p.numel() for p in model.parameters()):,}')
print(f'total_steps: {total_steps}  warmup: {warmup_steps}')

In [ ]:
best_f1       = 0.0
patience_left = PATIENCE
autocast_ctx  = torch.amp.autocast('cuda', dtype=torch.bfloat16) if DEVICE.type == 'cuda' else torch.amp.autocast('cpu')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()
    t0 = time.time()

    for step, batch in enumerate(train_loader, 1):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        with autocast_ctx:
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss   = criterion(logits, labels) / GRAD_ACCUM

        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM * labels.size(0)
        correct    += (logits.detach().float().argmax(1) == labels).sum().item()
        total      += labels.size(0)

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            with autocast_ctx:
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            all_preds  += logits.float().argmax(1).cpu().tolist()
            all_labels += labels.cpu().tolist()

    val_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels) * 100
    val_f1  = f1_score(all_labels, all_preds, average='macro') * 100

    print(f'epoch {epoch:>2}/{EPOCHS}  loss {total_loss/total:.4f}  acc {correct/total*100:.1f}%  '
          f'val_acc {val_acc:.1f}%  val_f1 {val_f1:.1f}%  ({time.time()-t0:.0f}s)')

    if val_f1 > best_f1:
        best_f1       = val_f1
        patience_left = PATIENCE
        torch.save(model.state_dict(), WEIGHTS_DIR / 'model.pt')
        model.save_pretrained(str(WEIGHTS_DIR))
        tokenizer.save_pretrained(str(WEIGHTS_DIR))
        print(f'  checkpoint saved  f1={val_f1:.2f}%')
    else:
        patience_left -= 1
        print(f'  no improvement  patience={patience_left}/{PATIENCE}')
        if patience_left == 0:
            print('  early stopping')
            break

print(f'best val_f1: {best_f1:.2f}%')

In [ ]:
import shutil

shutil.copy(PROCESSED_DIR / 'label_map.json', WEIGHTS_DIR / 'label_map.json')

for file_path in WEIGHTS_DIR.iterdir():
    if file_path.is_file():
        s3_key = f's3://{S3_BUCKET}/second/nlp/weights/{RUN}/{file_path.name}'
        os.system(f'aws s3 cp {file_path} {s3_key} --endpoint-url {S3_ENDPOINT}')
        print(f'{file_path.name} -> {s3_key}')

print('done')